# Grade 12 · Unit 1 — The ML workflow

> **SYNTHETIC** health table. Task: predict `institutional_delivery` (0/1). Findings say nothing about real women.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def data_file(dataset, filename):
    """Find datasets/<dataset>/clean/<filename> by walking up from this folder."""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        p = folder / "datasets" / dataset / "clean" / filename
        if p.exists():
            return p
    raise FileNotFoundError(f"Could not find {filename}. Open this notebook from inside the data-science-nepal folder.")

h = pd.read_csv(data_file("health-synthetic", "health_synthetic.csv"))
FEATURES = ["antenatal_visits", "wealth_quintile", "mother_age"]
X, y = h[FEATURES], h["institutional_delivery"]
print(h.shape, "| share delivering in a facility:", round(float(y.mean()), 3))

## 1. Split: 60% train, 20% validation, 20% test (test is locked until the end)

In [ ]:
from sklearn.model_selection import train_test_split
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=0.25, random_state=0, stratify=y_tmp)
assert len(X_train) + len(X_val) + len(X_test) == len(h)
print(len(X_train), len(X_val), len(X_test))

## 2. Baseline: always predict the most common class

In [ ]:
from sklearn.dummy import DummyClassifier
base = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print("baseline validation accuracy:", round(base.score(X_val, y_val), 3))

## 3. Train a model and compare on validation data

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
model = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_train, y_train)
val_acc = model.score(X_val, y_val)
print("model validation accuracy:", round(val_acc, 3))
assert 0 <= val_acc <= 1

## 4. Test ONCE at the end

In [ ]:
print("final test accuracy:", round(model.score(X_test, y_test), 3))

## Data leakage
**Predict:** what would happen if we scaled using all rows before splitting? Why is that a (small) leak? Write your answer, then try `make_pipeline` — it prevents it.

**Make:** write the task, baseline, model, and final test score in a 3-line summary for a non-expert.